# Cobrabox Example: Reciprocal Connectivity – #2 Analysis

Authors: *[COBRA group](https://cobra.cs.cas.cz), Institute of Computer Science, The Czech Academy of Sciences*

<div align="left">
<img src="Images/Logo_CAS_ICS.png" align="left" width="254" alt="logo ICS">
</div>


<br>
<br>

---------------------

> TODO: Fix from here

This notebook runs the full batch pipeline for all 20 subjects:
1. Extract notch-filtered bipolar segments and save to disk
2. Compute band-averaged connectivity matrices (PDC by default) and save
3. Compute Reciprocal Connectivity (RC) per band and save

Each step checks whether output already exists and skips if so, making it safe to re-run
after interruptions.


**Hypothesis:** Resected intracranial electrode contacts act as epileptic drivers and should
show higher Reciprocal Connectivity (RC > 0, net sink) compared to non-resected contacts,
as measured from interictal sleep iEEG.

This notebook loads pre-computed RC values, runs per-band statistical tests, and produces
visualisations.

## Import dependencies
> TODO: state the dependencies and point to documentation for installation.

This Notebook requires a ***python*** (>=3.11) installation together with ***xarray*** (>2026.2.0) and ***cobrabox*** (>=X.Y). For visualization it employs ***Matplotlib***. Please make sure these packages are installed in the python environment this notebook is running.


In [2]:
# Standard library imports
from pathlib import Path

# Third-party imports
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd  # Do we really need to import pandas? Can't do the same without Pandas?
import numpy as np
from scipy.stats import wilcoxon
import cobrabox as cb

# Local libraries and modules
from utils import *


Describe why next cell

In [3]:
# Some useful comment here
cb.set_dataset_dir(Path(".") / "data", persist=False)

# Some useful comment here
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 21)]
CONNECTIVITY_METHOD = "pdc"  # change here to try other methods

## 1. Load RC Values

For each subject and each frequency band we load the pre-computed RC DataArray and split
channels into two groups — **resected** and **non-resected** — using the patient metadata.
The result is a long-form DataFrame with one row per channel per band per subject.

In [ ]:
patient_info = load_patient_info()
band_names = list(BANDS.keys())

rows = []
for subject_id in SUBJECTS:
    rc_path = RC_DIR / f"{subject_id}_{CONNECTIVITY_METHOD}_rc.nc"
    if not rc_path.exists():
        print(f"WARNING: {subject_id} RC not found, skipping")
        continue

    rc_by_band = load_rc(subject_id, method=CONNECTIVITY_METHOD)
    resected_pairs = set(patient_info[subject_id]["resected"])

    for band_name, rc_da in rc_by_band.items():
        channels = rc_da.coords["space"].values
        for ch in channels:
            group = "resected" if ch in resected_pairs else "non_resected"
            rows.append({
                "subject": subject_id,
                "band": band_name,
                "channel": ch,
                "group": group,
                "rc_value": float(rc_da.sel(space=ch).values),
            })

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
print()
print(df.head(10).to_string(index=False))

## 2. Per-Subject Mean RC

Statistical tests comparing resected vs non-resected must be performed at the
**subject level** (not at the channel level) to avoid pseudoreplication — channels
within the same subject are not independent. We collapse to one mean RC value
per subject per band per group.

In [ ]:
df_sub = (
    df.groupby(["subject", "band", "group"])["rc_value"]
    .mean()
    .reset_index()
)

df_means = df_sub.pivot_table(
    index=["subject", "band"],
    columns="group",
    values="rc_value",
).reset_index()
df_means.columns.name = None

if "resected" not in df_means.columns:
    df_means["resected"] = np.nan
if "non_resected" not in df_means.columns:
    df_means["non_resected"] = np.nan

df_means["diff"] = df_means["resected"] - df_means["non_resected"]
df_means = df_means.rename(columns={"resected": "rc_resected", "non_resected": "rc_non_resected"})

print(f"Per-subject mean RC shape: {df_means.shape}")
print()
print(df_means.head(10).to_string(index=False))

## 3. Statistical Tests

We use the **Wilcoxon signed-rank test** (non-parametric, paired) to test whether the
per-subject mean RC of resected channels is significantly greater than that of non-resected
channels. The test is **one-sided** (`alternative='greater'`) consistent with our directional
hypothesis. Significance threshold: p < 0.05.

In [ ]:
stat_rows = []
for band_name in band_names:
    band_df = df_means[df_means["band"] == band_name].dropna(
        subset=["rc_resected", "rc_non_resected"]
    )
    res_vals = band_df["rc_resected"].values
    non_vals = band_df["rc_non_resected"].values
    diffs = band_df["diff"].values

    if len(res_vals) < 2:
        stat_rows.append({"band": band_name, "n_subjects": len(res_vals),
                          "mean_diff": np.nan, "median_diff": np.nan,
                          "W": np.nan, "p_value": np.nan, "significant": False})
        continue

    try:
        W, p = wilcoxon(res_vals, non_vals, alternative="greater")
    except ValueError:
        W, p = np.nan, np.nan

    stat_rows.append({
        "band": band_name,
        "n_subjects": len(res_vals),
        "mean_diff": float(np.mean(diffs)),
        "median_diff": float(np.median(diffs)),
        "W": W,
        "p_value": p,
        "significant": p < 0.05 if not np.isnan(p) else False,
    })

stats_df = pd.DataFrame(stat_rows)
pd.set_option("display.float_format", "{:.4f}".format)
print(stats_df.to_string(index=False))

## 4. Visualization: Paired Dot Plots

Each panel shows one frequency band. Gray lines connect the same subject's resected and
non-resected mean RC. The bold black horizontal lines mark the cross-subject mean for each
group. A consistent upward trend from non-resected (blue) to resected (red) would support
the hypothesis.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, band_name in zip(axes, band_names):
    band_df = df_means[df_means["band"] == band_name].dropna(
        subset=["rc_resected", "rc_non_resected"]
    )
    res_vals = band_df["rc_resected"].values
    non_vals = band_df["rc_non_resected"].values

    for r, n in zip(res_vals, non_vals):
        ax.plot([0, 1], [n, r], color="gray", alpha=0.4, lw=0.8)

    ax.scatter(np.zeros(len(non_vals)), non_vals, color="steelblue", zorder=3, s=25, label="non-resected")
    ax.scatter(np.ones(len(res_vals)), res_vals, color="tomato", zorder=3, s=25, label="resected")

    ax.plot([0, 1], [np.mean(non_vals), np.mean(res_vals)],
            color="black", lw=2.5, zorder=4)

    p_row = stats_df[stats_df["band"] == band_name]
    if len(p_row) > 0:
        p_val = p_row["p_value"].values[0]
        sig = "*" if p_row["significant"].values[0] else ""
        p_str = f"p={p_val:.3f}{sig}" if not np.isnan(p_val) else "p=N/A"
    else:
        p_str = ""

    ax.set_title(f"{band_name}\n{p_str}", fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["non-res.", "resected"], fontsize=8)
    ax.set_ylabel("mean RC", fontsize=8)

handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=7, label="non-resected"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="tomato", markersize=7, label="resected"),
]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Reciprocal Connectivity: Resected vs Non-Resected", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. Band Comparison

Which frequency band shows the strongest and most consistent RC difference between
resected and non-resected channels? Here we plot the mean difference (resected − non-resected)
per band, sorted by effect size. Bars are filled for significant bands (p < 0.05) and
hatched for non-significant ones. Error bars show the standard error of the mean (SEM)
across subjects.

In [ ]:
plot_df = stats_df.copy().dropna(subset=["mean_diff"])

sem_by_band = {}
for band_name in band_names:
    band_diffs = df_means[df_means["band"] == band_name]["diff"].dropna().values
    sem_by_band[band_name] = float(np.std(band_diffs) / np.sqrt(len(band_diffs))) if len(band_diffs) > 1 else 0.0

plot_df = plot_df.sort_values("mean_diff", ascending=False).reset_index(drop=True)
plot_df["sem"] = plot_df["band"].map(sem_by_band)

colors = ["steelblue" if sig else "lightsteelblue" for sig in plot_df["significant"]]
hatches = ["" if sig else "//" for sig in plot_df["significant"]]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(
    plot_df["band"],
    plot_df["mean_diff"],
    color=colors,
    edgecolor="black",
    linewidth=0.8,
    yerr=plot_df["sem"],
    capsize=4,
    error_kw={"elinewidth": 1.2},
)
for bar, hatch in zip(bars, hatches):
    bar.set_hatch(hatch)

ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("Frequency band")
ax.set_ylabel("Mean RC difference (resected − non-resected)")
ax.set_title("Mean RC difference (resected − non-resected) by band")

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="steelblue", edgecolor="black", label="p < 0.05"),
    Patch(facecolor="lightsteelblue", edgecolor="black", hatch="//", label="p ≥ 0.05"),
]
ax.legend(handles=legend_elements, fontsize=9)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Summary

Key findings from the RC analysis across 20 subjects:

- *[Which bands, if any, showed significantly higher RC in resected vs non-resected channels? Report p-values and effect sizes.]*
- *[Was the effect consistent across subjects (most lines going upward in paired plots) or driven by outliers?]*
- *[Which band showed the largest mean difference? Does this align with the expected frequency range of epileptic activity (e.g. gamma, ripples)?]*
- *[Overall: does the evidence support the hypothesis that RC from PDC can identify resected channels from interictal iEEG?]*